# Advance Settlement Automation & AI Reconciliation Walkthrough
### PT. Setiawan Dwi Tunggal (SouthCity) — IT & Finance Department

Notebook ini menyajikan langkah demi langkah otomatisasi pemadanan (*reconciliation*) transaksi **Uang Muka (Advances - Other)** antara data **General Ledger (GL)** di SQL Server dan **Working Paper** monitoring Finance untuk periode April 2026, dilengkapi integrasi **Google Gemini AI** untuk analisis eksekutif.

In [1]:
import os
import sys
import pandas as pd
import openpyxl
from pathlib import Path

# Deteksi jika dijalankan di Google Colab (Online Run)
if "google.colab" in sys.modules:
    if not os.path.exists("automation"):
        !git clone https://github.com/IkhwanMaulana07/automation.git
    %cd automation
    !pip install -r requirements.txt -q
    sys.path.append(os.getcwd())
else:
    ROOT_DIR = Path("..").resolve()
    if str(ROOT_DIR) not in sys.path:
        sys.path.append(str(ROOT_DIR))

from src.matcher import ReconciliationMatcher
from src.ai_agent import ExecutiveSummaryAgent
from src.sheets_sync import DualSheetExporter
from src.config import GL_FILE_PATH, WP_FILE_PATH, OUTPUT_EXCEL_PATH, GEMINI_API_KEY

print("Environment and dependencies loaded successfully!")

Dependencies loaded successfully!


## 1. Eksplorasi Data Mentah (Raw Data Ingestion)
Melihat data transaksi kredit GL dan item Uang Muka di Working Paper.

In [2]:
# 1. Baca GL Transaksi Kredit
gl_df = pd.read_excel(GL_FILE_PATH)
gl_kredit = gl_df[gl_df['KREDIT-IDR'] > 0][['TANGGAL', 'NO JURNAL', 'DESKRIPSI', 'KREDIT-IDR']].copy()
print(f"Total Transaksi Kredit GL: {len(gl_kredit)} baris")
gl_kredit.head(10)

Total Transaksi Kredit GL: 39 baris


,TANGGAL,NO JURNAL,DESKRIPSI,KREDIT-IDR
1,2026-04-01,BCA2/BK/2604/0026,PELUNASAN TAGIHAN DROPBOX PERIODE 11/3/2026 - ...,60000000
2,2026-04-01,BCA2/BM/2604/0003,PENGEMBALIAN ADVANCE PT SEM UNTUK PEMBELIAN DI...,2330500
3,2026-04-02,ADV/BK/2604/0004,TP01/PO/26010005 FURNITURE SHOW UNIT APARTEMEN...,2500000
4,2026-04-02,ADV/BK/2604/0005,FR01/PO/26010009 PAKET MEETING DI KIRANA RESTO...,1750000
7,2026-04-02,PMT2/BK/2604/0006,TRANSFER KEKURANGAN DANA UM ACARA BUKA PUASA B...,3500000
9,2026-04-07,ADV/BK/2604/0001,HLJC/PO/26030002 SERAGAM TENAGA KERJA HARIAN,3780000
10,2026-04-07,ADV/BK/2604/0006,FR01/PO/26020012 SEWA SPACE OPEN TABLE LOKASI ...,4202100
11,2026-04-07,ADV/BK/2604/0009,HLJC/PO/26030003 BORDIR SERAGAM TENAGA KERJA H...,2790000
12,2026-04-08,PMT2/BM/2604/0006,PENGEMBALIAN KELEBIHAN DANA UM COMPLIMENTARY S...,2602000
13,2026-04-09,ADV/BK/2604/0003,FR01/PO/26010007 EO PAMERAN DI THE PARK SAWANG...,25070000


In [3]:
# 2. Baca Working Paper Input (Baris 8-22)
wb = openpyxl.load_workbook(WP_FILE_PATH, data_only=True)
ws = wb.active
wp_data = []
for r in range(8, 23):
    wp_data.append({
        'Row': r,
        'Date': ws.cell(r, 1).value,
        'Voucher No': ws.cell(r, 2).value,
        'Description': ws.cell(r, 3).value,
        'Amount': ws.cell(r, 4).value
    })
wp_df = pd.DataFrame(wp_data)
print(f"Total Item Uang Muka di Working Paper: {len(wp_df)} baris")
wp_df

Total Item Uang Muka di Working Paper: 15 baris


,Row,Date,Voucher No,Description,Amount
0,8,2026-01-22,PMT2/BK/2601/0062,TP01/PO/26010005 FURNITURE SHOW UNIT APARTEMEN...,2500000.0
1,9,2026-01-22,PMT2/BK/2601/0064,STYLING APARTEMEN THE PARC SM 1132 (2 BEDROOM),25695900.0
2,10,2026-01-22,PMT2/BK/2601/0065,STYLING APARTEMEN THE PARC SM 1127 (STUDIO),1583700.0
3,11,2026-01-22,PMT2/BK/2601/0068,PAKET MEETING DI KIRANA RESTO & CAFÉ,1750000.0
4,12,2026-02-05,PMT2/BK/2602/0003,TALENTA - BUSSINES FLAT FOR 4 MONTHS (TERM-3) ...,10355000.0
5,13,2026-02-05,PMT2/BK/2602/0005,FR01/PO/26010011 PENULIS WIKIPEDIA INDONESIA,304000.0
6,14,2026-02-12,PMT2/BK/2602/0071,FR01/PO/26010007 EO PAMERAN DI THE PARK SAWANG...,25070000.0
7,15,2026-02-25,PMT2/BK/2602/0092,PEMBAYARAN PBB JV 2 SUMMARECON TAHUN 2026,422974179.0
8,16,2026-03-05,PMT2/BK/2603/0008,HLJC/PO/26030003 BORDIR SERAGAM TENAGA KERJA H...,2790000.0
9,17,2026-03-05,PMT2/BK/2603/0009,HLJC/PO/26030002 SERAGAM TENAGA KERJA HARIAN,3780000.0


## 2. Eksekusi Matching Engine (Multi-Tier Algorithm)
Algoritma melakukan pemadanan berdasarkan:
1. **Tier 1: Regex Nomor PO/WO** (`TP01/PO/...`, `HLJC/PO/...`, dll).
2. **Tier 2: Kesamaan Frasa Proyek** (`Styling Apartemen`, `Dropbox`, `Talenta`, `PBB JV 2`, dll).
3. **Multi-Voucher Aggregator:** Menjumlahkan penyelesaian multi-voucher (seperti Styling 2BR & Studio).
4. **Kalkulasi Sisa Saldo:** $\text{Saldo} = \text{Amount} - \text{Realization Amount}$.

In [4]:
matcher = ReconciliationMatcher(GL_FILE_PATH, WP_FILE_PATH)
results = matcher.match_all()
metrics = matcher.get_summary_metrics()

res_df = pd.DataFrame(results)[[
    'advance_voucher', 'desc', 'amount', 'realization_date', 
    'realization_voucher', 'realization_amount', 'saldo', 'status'
]]
res_df

,advance_voucher,desc,amount,realization_date,realization_voucher,realization_amount,saldo,status
0,PMT2/BK/2601/0062,TP01/PO/26010005 FURNITURE SHOW UNIT APARTEMEN...,2500000.0,2026-04-02,ADV/BK/2604/0004,2500000.0,0.0,SETTLED
1,PMT2/BK/2601/0064,STYLING APARTEMEN THE PARC SM 1132 (2 BEDROOM),25695900.0,2026-04-17,"ADV/BK/2604/0012, ADV/BK/2604/0013, ADV/BK/260...",25695900.0,0.0,SETTLED
2,PMT2/BK/2601/0065,STYLING APARTEMEN THE PARC SM 1127 (STUDIO),1583700.0,2026-04-17,"ADV/BK/2604/0019, ADV/BK/2604/0020, PMT2/BM/26...",1583700.0,0.0,SETTLED
3,PMT2/BK/2601/0068,PAKET MEETING DI KIRANA RESTO & CAFÉ,1750000.0,2026-04-02,ADV/BK/2604/0005,1750000.0,0.0,SETTLED
4,PMT2/BK/2602/0003,TALENTA - BUSSINES FLAT FOR 4 MONTHS (TERM-3) ...,10355000.0,2026-04-17,ADV/BK/2604/0008,10355000.0,0.0,SETTLED
5,PMT2/BK/2602/0005,FR01/PO/26010011 PENULIS WIKIPEDIA INDONESIA,304000.0,2026-04-09,ADV/BK/2604/0007,304000.0,0.0,SETTLED
6,PMT2/BK/2602/0071,FR01/PO/26010007 EO PAMERAN DI THE PARK SAWANG...,25070000.0,2026-04-09,ADV/BK/2604/0003,25070000.0,0.0,SETTLED
7,PMT2/BK/2602/0092,PEMBAYARAN PBB JV 2 SUMMARECON TAHUN 2026,422974179.0,2026-04-23,BCA1/BM/2604/0069,315869963.0,107104216.0,PARTIAL
8,PMT2/BK/2603/0008,HLJC/PO/26030003 BORDIR SERAGAM TENAGA KERJA H...,2790000.0,2026-04-07,ADV/BK/2604/0009,2790000.0,0.0,SETTLED
9,PMT2/BK/2603/0009,HLJC/PO/26030002 SERAGAM TENAGA KERJA HARIAN,3780000.0,2026-04-07,ADV/BK/2604/0001,3780000.0,0.0,SETTLED


## 3. Analisis Khusus: Multi-Voucher & Item Unsettled
### A. Detail Multi-Voucher Styling Apartemen The Parc SM 1132 (2 Bedroom)
Terdiri dari 8 voucher GL kredit (7 belanja furniture/styling + 1 retur kas kelebihan dana) yang pas 100% senilai Rp 25.695.900.

In [5]:
styling_item = next(r for r in results if 'SM 1132' in r['desc'] and 'STYLING' in r['desc'])
styling_vouchers = pd.DataFrame(styling_item['breakdown'])[['date', 'voucher', 'amount', 'desc', 'reason']]
print(f"Total Nominal Realisasi: Rp {styling_vouchers['amount'].sum():,.0f} (Target WP: Rp {styling_item['amount']:,.0f})")
styling_vouchers

Total Nominal Realisasi: Rp 25,695,900 (Target WP: Rp 25,695,900)


,date,voucher,amount,desc,reason
0,2026-04-17,ADV/BK/2604/0012,7867700.0,TP01/PO/26030008 FURNITURE & STYLING SHOW UNIT...,Phrase: Styling SM 1132 (2BR)
1,2026-04-17,ADV/BK/2604/0013,8001000.0,TP01/PO/26030005 STYLING APARTEMEN THE PARC SM...,Phrase: Styling SM 1132 (2BR)
2,2026-04-17,ADV/BK/2604/0014,1006000.0,TP01/PO/26040005 STYLING APARTEMEN THE PARC SM...,Phrase: Styling SM 1132 (2BR)
3,2026-04-17,ADV/BK/2604/0015,729000.0,TP01/PO/26040002 STYLING APARTEMEN THE PARC SM...,Phrase: Styling SM 1132 (2BR)
4,2026-04-17,ADV/BK/2604/0016,761900.0,TP01/PO/26040004 STYLING APARTEMEN THE PARC SM...,Phrase: Styling SM 1132 (2BR)
5,2026-04-17,ADV/BK/2604/0017,1610000.0,TP01/PO/26040003 STYLING APARTEMEN THE PARC SM...,Phrase: Styling SM 1132 (2BR)
6,2026-04-17,ADV/BK/2604/0018,3692000.0,TP01/PO/26040006 STYLING APARTEMEN THE PARC SM...,Phrase: Styling SM 1132 (2BR)
7,2026-04-17,PMT2/BM/2604/0007,2028300.0,PENGEMBALIAN KELEBIHAN DANA UM STYLING APARTEM...,Phrase: Styling SM 1132 (2BR)


### B. Detail Transaksi Parsial: PBB JV 2 Summarecon
Uang muka sebesar Rp 422.974.179 diselesaikan sebagian oleh voucher `BCA1/BM/2604/0069` senilai Rp 315.869.963, menyisakan saldo Rp 107.104.216.

In [6]:
pbb_item = next(r for r in results if 'PBB' in r['desc'])
pd.DataFrame([{
    'Deskripsi': pbb_item['desc'],
    'Advance Awal': f"Rp {pbb_item['amount']:,.0f}",
    'Realisasi GL': f"Rp {pbb_item['realization_amount']:,.0f}",
    'Sisa Saldo Terbuka': f"Rp {pbb_item['saldo']:,.0f}",
    'No Voucher GL': pbb_item['realization_voucher'],
    'Tgl Realisasi': pbb_item['realization_date']
}])

,Deskripsi,Advance Awal,Realisasi GL,Sisa Saldo Terbuka,No Voucher GL,Tgl Realisasi
0,PEMBAYARAN PBB JV 2 SUMMARECON TAHUN 2026,"Rp 422,974,179","Rp 315,869,963","Rp 107,104,216",BCA1/BM/2604/0069,2026-04-23


## 4. Integrasi AI (Executive Summary Generator)
Memanfaatkan **Google Gemini API** (`gemini-2.5-flash`) untuk menganalisis temuan rekonsiliasi dan menyusun ringkasan eksekutif beserta rekomendasi tindak lanjut bagi Direksi.

In [7]:
from IPython.display import display, Markdown

ai_agent = ExecutiveSummaryAgent(api_key=GEMINI_API_KEY)
summary_md = ai_agent.generate_summary(metrics, results)
display(Markdown(summary_md))

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Tentu, berikut adalah Executive Summary profesional dan ringkas mengenai hasil rekonsiliasi Advance Settlement periode April 2026 untuk Direksi PT. Setiawan Dwi Tunggal (SouthCity).

---

**EXECUTIVE SUMMARY**
**Rekonsiliasi Advance Settlement Periode April 2026**

**Kepada Yth. Direksi PT. Setiawan Dwi Tunggal (SouthCity)**
**Dari:** Senior Finance & Automation Controller
**Tanggal:** 02 Mei 2026

Laporan ini menyajikan ringkasan hasil rekonsiliasi Advance Settlement untuk periode April 2026, mencakup kinerja penyelesaian, sorotan transaksi kunci, dan rekomendasi tindak lanjut.

---

### 1. Ringkasan Eksekutif & Kinerja Settlement

Rekonsiliasi Advance Settlement untuk periode April 2026 telah selesai dilaksanakan. Secara keseluruhan, kinerja penyelesaian uang muka menunjukkan efisiensi yang baik dengan rasio penyelesaian yang tinggi.

*   **Total Nilai Uang Muka (Advance) Awal:** Rp 570.406.879 (dari 15 transaksi)
*   **Total Realisasi / Settlement Masuk:** Rp 463.302.663
*   **Total Sisa Saldo Outstanding:** Rp 107.104.216
*   **Rasio Penyelesaian (Settlement Rate):** 81,22%
*   **Jumlah Transaksi Selesai (Fully Settled):** 14 dari 15 transaksi

Tingkat penyelesaian sebesar 81,22% dan 14 dari 15 transaksi telah diselesaikan penuh menunjukkan pengelolaan advance yang efektif, dengan hanya satu transaksi yang masih memiliki saldo terbuka.

### 2. Sorotan Transaksi Kritis & Item Unsettled

Satu-satunya transaksi yang masih memiliki sisa saldo terbuka dan menjadi fokus utama adalah:

*   **Nama Transaksi:** PEMBAYARAN PBB JV 2 SUMMARECON TAHUN 2026
*   **Voucher Pengajuan Advance:** PMT2/BK/2602/0092
*   **Nilai Advance Awal:** Rp 422.974.179
*   **Realisasi Diterima (Parsial):** Rp 315.869.963 (melalui voucher BCA1/BM/2604/0069 - Penerimaan Pembayaran PBB MSCM AJB JV 2)
*   **Sisa Saldo Terbuka:** Rp 107.104.216

Sisa saldo ini merupakan hasil dari pelunasan parsial atas pembayaran PBB JV 2 Summarecon, di mana sebagian dana telah diterima kembali, namun masih terdapat selisih yang belum terselesaikan.

### 3. Penyelesaian Multi-Voucher Sukses

Kami juga ingin menyoroti keberhasilan penyelesaian 100% atas transaksi **Multi-Voucher Styling Apartemen (2BR & Studio)**. Rekonsiliasi untuk transaksi ini, yang melibatkan kombinasi belanja vendor dan retur pengembalian kelebihan dana kas, telah diselesaikan secara akurat dan komprehensif. Hal ini menunjukkan kemampuan tim dalam menangani transaksi yang kompleks dengan berbagai sumber penyelesaian.

### 4. Rekomendasi Tindak Lanjut (Action Plan)

Berdasarkan hasil rekonsiliasi ini, kami merekomendasikan langkah-langkah berikut:

1.  **Tindak Lanjut Saldo PBB JV 2 Summarecon:** Segera lakukan koordinasi dan tindak lanjut dengan departemen terkait atau mitra JV untuk mengklarifikasi dan menyelesaikan sisa saldo Rp 107.104.216 dari transaksi PBB JV 2 Summarecon. Target penyelesaian diharapkan dalam 2 minggu ke depan.
2.  **Analisis Akar Masalah (Opsional):** Identifikasi penyebab sisa saldo PBB JV 2 Summarecon untuk mencegah kejadian serupa di masa mendatang, jika memungkinkan, atau untuk memastikan proses settlement di masa depan lebih efisien.
3.  **Pemantauan Berkelanjutan:** Terus tingkatkan pemantauan dan percepatan proses settlement untuk menjaga rasio penyelesaian yang tinggi dan meminimalkan saldo outstanding secara proaktif.

---

Demikian Executive Summary ini kami sampaikan untuk menjadi perhatian dan tindak lanjut Direksi.

Hormat kami,

**[Nama Anda]**
Senior Finance & Automation Controller
PT. Setiawan Dwi Tunggal (SouthCity)

## 5. Export ke Excel Dual-Sheet & Google Sheets
Menghasilkan file Excel final dengan 2 tab: `Dashboard` dan `Working_Paper_Result`.

In [8]:
# Simpan Excel
matcher.generate_excel_result(str(OUTPUT_EXCEL_PATH))
exporter = DualSheetExporter(str(OUTPUT_EXCEL_PATH))
exporter.add_dashboard_sheet(metrics, summary_md)
print(f"File Excel tersimpan di: {OUTPUT_EXCEL_PATH.resolve()}")

File Excel tersimpan di: C:\automation-skilltes\output\Working_Paper_Result.xlsx
